# Preprocessing & Binary Anxiety Classification with STAI Questionnaire Labels

This notebook preprocesses WESAD signals and extracts 60-second windows every 6 seconds. For each participant, it reverse-scores the WESAD shortened STAI items (items 1, 4, and 6) and averages the resulting valid responses by condition and labels a non-Base condition Anxiety (`1`) when its mean is at least 0.2 points above that participantâ€™s Base mean; otherwise it labels No Anxiety (`0`). This is an operational, baseline-relative self-report label: it does not assume that TSST must increase every participantâ€™s STAI score. Whether TSST reliably increases STAI is a separate empirical hypothesis to evaluate from the participant-level scores. Only windows fully contained in one timed, labeled condition are retained. Existing `S*_stai_60s_shift6s.pkl` files must be regenerated after changing the labeling logic or threshold.

**Sampling-rate alignment:** non-ACC modalities are resampled to 70 Hz before they are aligned to the 700 Hz WESAD label clock. Wrist ACC remains at 32 Hz and chest ACC remains at 700 Hz so ACC features are not downsampled. The 700 Hz clock is retained only for labels, window boundaries, and timestamp alignment.

**ACC sub-window variant:** each retained 60-second main window also computes ACC-specific features over 5-second sub-windows shifted by 0.25 seconds. The resulting ACC statistics are aggregated using the mean and standard deviation across sub-windows, while all non-ACC signals keep the original 60-second processing window.


In [40]:
import gc
import pickle
import sys
import time
import warnings
from pathlib import Path
from unittest.mock import MagicMock

import matplotlib
matplotlib.use('Agg')
import numpy as np
import pandas as pd
from scipy.signal import resample_poly
import heartpy as hp
import neurokit2 as nk

warnings.simplefilter('ignore')

# Prevent biosppy's optional interactive plotting modules from requiring tkinter.
sys.modules['biosppy.inter_plotting'] = MagicMock()
sys.modules['biosppy.inter_plotting.ecg'] = MagicMock()
sys.modules['biosppy.inter_plotting.acc'] = MagicMock()
import biosppy

REPO_ROOT = next(p for p in [Path.cwd(), *Path.cwd().parents] if (p / 'WESAD').exists())
FB_CODE_CANDIDATES = [
    REPO_ROOT / 'Scripts' / '3rd try' / 'fb_code',
    REPO_ROOT / '_forked' / 'fb_code',
    Path.cwd() / 'fb_code',
]
FB_CODE_DIR = next(
    path for path in FB_CODE_CANDIDATES
    if (path / 'respiration.py').exists() and (path / 'utils.py').exists()
)
if str(FB_CODE_DIR) not in sys.path:
    sys.path.insert(0, str(FB_CODE_DIR))

from utils import (
    butter_lowpass_filter,
    filterSignalFIR as filter_signal_fir,
    get_net_accel,
    get_window_stats,
    get_peak_freq,
    get_absolute_integral,
    get_slope,
    get_dynamic_range,
)
from respiration import get_resp_features

# User-configurable pipeline settings.
SUBJECTS = [2, 3, 4, 5, 6, 7, 8, 9, 10, 11, 13, 14, 15, 16, 17]
WINDOW_LENGTH_SECONDS = 60
WINDOW_SHIFT_SECONDS = 6  # fourth-try requirement
ACC_WINDOW_LENGTH_SECONDS = 5
ACC_WINDOW_SHIFT_SECONDS = 0.25
NON_ACC_FS = 70
STAI_INCREASE_THRESHOLD = 0.2  # anxiety if condition mean - personal Base mean >= this
SKIP_EXTRACTED_SUBJECTS = False  # existing files contain old protocol-derived labels
OVERWRITE_EXISTING_OUTPUTS = False  # set False to refuse replacing existing files

# RAW_FS describes the WESAD files; FS describes the feature-extraction rates.
RAW_FS = {'ACC': 32, 'BVP': 64, 'EDA': 4, 'TEMP': 4, 'label': 700,
          'Resp': 700, 'ECG': 700, 'chest': 700}
FS = {'ACC': 32, 'BVP': NON_ACC_FS, 'EDA': NON_ACC_FS, 'TEMP': NON_ACC_FS, 'label': 700,
      'Resp': NON_ACC_FS, 'ECG': NON_ACC_FS, 'chest': 700}
WESAD_DIR = REPO_ROOT / 'WESAD'
OUTPUT_DIR = REPO_ROOT / 'Data' / 'Preprocessed n STAI-labeled 6s'
OUTPUT_DIR.mkdir(parents=True, exist_ok=True)
MODELS_DIR = REPO_ROOT / 'Models'
MODELS_DIR.mkdir(parents=True, exist_ok=True)

if WINDOW_LENGTH_SECONDS <= 0 or WINDOW_SHIFT_SECONDS <= 0:
    raise ValueError('WINDOW_LENGTH_SECONDS and WINDOW_SHIFT_SECONDS must be positive.')
if ACC_WINDOW_LENGTH_SECONDS <= 0 or ACC_WINDOW_SHIFT_SECONDS <= 0:
    raise ValueError('ACC_WINDOW_LENGTH_SECONDS and ACC_WINDOW_SHIFT_SECONDS must be positive.')
if ACC_WINDOW_LENGTH_SECONDS > WINDOW_LENGTH_SECONDS:
    raise ValueError('ACC_WINDOW_LENGTH_SECONDS cannot exceed WINDOW_LENGTH_SECONDS.')
print('Setup completed:', OUTPUT_DIR)
print(f'Window length={WINDOW_LENGTH_SECONDS}s, shift={WINDOW_SHIFT_SECONDS}s')
print(f'ACC sub-window length={ACC_WINDOW_LENGTH_SECONDS}s, shift={ACC_WINDOW_SHIFT_SECONDS}s')
print(f'STAI increase threshold={STAI_INCREASE_THRESHOLD}')

Setup completed: e:\Rizqy\Kuliah\Skripsi\repo-skripsi\Data\Preprocessed n STAI-labeled 6s
Window length=60s, shift=6s
ACC sub-window length=5s, shift=0.25s
STAI increase threshold=0.2


## 1. Questionnaire & STAI Parser

In [41]:
def parse_clock_mmss(value):
    """Convert questionnaire MM.SS notation (e.g. 7.08 -> 428 seconds)."""
    text = str(value).strip()
    if not text:
        return np.nan
    if '.' in text:
        minute_text, second_text = text.split('.', 1)
        minutes = int(minute_text)
        seconds = int(second_text)
    else:
        minutes, seconds = int(text), 0
    if not 0 <= seconds < 60:
        raise ValueError(f'Invalid MM.SS time {value!r}: seconds must be 00-59')
    return minutes * 60 + seconds


def _parse_numeric_fields(parts):
    """Preserve CSV field alignment, converting empty fields to NaN."""
    values = []
    for part in parts:
        text = part.strip()
        if not text:
            values.append(np.nan)
        else:
            try:
                values.append(float(text))
            except ValueError:
                values.append(np.nan)
    return values


# WESAD uses a shortened six-item state-anxiety questionnaire. The positively
# worded items are reverse-scored before averaging: 1 <-> 4 and 2 <-> 3.
STAI_ITEM_NAMES = (
    'feel_at_ease', 'feel_nervous', 'jittery', 'relaxed', 'worried', 'pleasant'
)
STAI_REVERSE_ITEM_INDICES = {0, 3, 5}


def score_stai_responses(raw_responses):
    """Return raw and correctly reverse-scored six-item STAI responses."""
    raw = [float(value) if pd.notna(value) else np.nan for value in raw_responses]
    if len(raw) > len(STAI_ITEM_NAMES):
        raise ValueError(f'Expected at most {len(STAI_ITEM_NAMES)} shortened STAI items, got {len(raw)}')
    scored = [
        (5.0 - value) if index in STAI_REVERSE_ITEM_INDICES and pd.notna(value) else value
        for index, value in enumerate(raw)
    ]
    return raw, scored


def parse_quest_file(subject_id):
    quest_path = WESAD_DIR / f'S{subject_id}' / f'S{subject_id}_quest.csv'
    if not quest_path.exists():
        raise FileNotFoundError(f'Questionnaire file not found: {quest_path}')

    order = start_raw = end_raw = None
    stai_rows = []
    with open(quest_path, 'r', encoding='latin1') as f:
        for line in f:
            parts = [part.strip() for part in line.rstrip('\r\n').split(';')]
            tag = parts[0].strip() if parts else ''
            if tag.startswith('# ORDER'):
                order = [value.strip() for value in parts[1:]]
            elif tag.startswith('# START'):
                start_raw = parts[1:]
            elif tag.startswith('# END'):
                end_raw = parts[1:]
            elif tag.startswith('# STAI'):
                stai_rows.append(_parse_numeric_fields(parts[1:]))

    if order is None or start_raw is None or end_raw is None:
        raise ValueError(f'{quest_path.name}: missing ORDER, START, or END row')
    if len(start_raw) < len(order) or len(end_raw) < len(order):
        raise ValueError(f'{quest_path.name}: timing rows do not cover all conditions')
    if not stai_rows:
        raise ValueError(f'{quest_path.name}: no STAI response rows found')

    conditions = []
    for index, condition in enumerate(order):
        if not condition:
            continue
        start_text, end_text = start_raw[index].strip(), end_raw[index].strip()
        start_seconds = parse_clock_mmss(start_text) if start_text else np.nan
        end_seconds = parse_clock_mmss(end_text) if end_text else np.nan
        responses = [row[index] if index < len(row) else np.nan for row in stai_rows]
        raw_responses, scored_responses = score_stai_responses(responses)
        valid_raw = [float(value) for value in raw_responses if pd.notna(value)]
        valid_scored = [float(value) for value in scored_responses if pd.notna(value)]
        if any(value < 1 or value > 4 for value in valid_raw):
            raise ValueError(f'{quest_path.name}: STAI response outside 1-4 for condition {condition!r}')
        conditions.append({
            'condition': condition,
            'start_seconds': start_seconds,
            'end_seconds': end_seconds,
            'stai_responses_raw': valid_raw,
            'stai_responses_scored': valid_scored,
            'stai_item_count': len(valid_scored),
            'stai_mean_raw': float(np.mean(valid_raw)) if valid_raw else np.nan,
            'stai_mean': float(np.mean(valid_scored)) if valid_scored else np.nan,
        })

    if not any(item['condition'].strip().lower() == 'base' for item in conditions):
        raise ValueError(f'{quest_path.name}: no Base condition found')
    return conditions


# Inspect parsed condition data for every configured participant.
all_participant_conditions = []
for subject_id in SUBJECTS:
    participant_df = pd.DataFrame(parse_quest_file(subject_id))
    participant_df.insert(0, 'subject_id', subject_id)
    all_participant_conditions.append(participant_df)

all_conditions_df = pd.concat(all_participant_conditions, ignore_index=True)
display(all_conditions_df.sort_values(['subject_id', 'condition']).reset_index(drop=True))


,subject_id,condition,start_seconds,end_seconds,stai_responses_raw,stai_responses_scored,stai_item_count,stai_mean_raw,stai_mean
0,2,Base,428,1592,"[3.0, 1.0, 4.0, 3.0, 4.0]","[2.0, 1.0, 4.0, 2.0, 4.0]",5,3.0,2.6
1,2,Fun,4885,5267,"[4.0, 1.0, 4.0, 3.0, 4.0]","[1.0, 1.0, 4.0, 2.0, 4.0]",5,3.2,2.4
2,2,Medi 1,4219,4621,"[1.0, 2.0, 1.0, 1.0, 1.0]","[4.0, 2.0, 1.0, 4.0, 1.0]",5,1.2,2.4
3,2,Medi 2,5618,6015,"[2.0, 3.0, 2.0, 1.0, 1.0]","[3.0, 3.0, 2.0, 4.0, 1.0]",5,1.8,2.6
4,2,TSST,2395,3003,"[2.0, 3.0, 1.0, 1.0, 1.0]","[3.0, 3.0, 1.0, 4.0, 1.0]",5,1.6,2.4
...,...,...,...,...,...,...,...,...,...
114,17,Medi 2,5349,5763,"[1.0, 1.0, 1.0, 4.0, 1.0]","[4.0, 1.0, 1.0, 1.0, 1.0]",5,1.6,1.6
115,17,TSST,3602,4345,"[2.0, 4.0, 3.0, 1.0, 2.0]","[3.0, 4.0, 3.0, 4.0, 2.0]",5,2.4,3.2
116,17,bRead,1417,1495,"[2.0, 4.0, 2.0, 1.0, 1.0]","[3.0, 4.0, 2.0, 4.0, 1.0]",5,2.0,2.8
117,17,fRead,2282,2359,[],[],0,NaN,NaN


## 2. Signal Preprocessing & Binary STAI Labeling Pipeline

In [42]:
def load_raw_subject(subject_id):
    path = WESAD_DIR / f'S{subject_id}' / f'S{subject_id}.pkl'
    with open(path, 'rb') as file:
        return pickle.load(file, encoding='latin1')


def build_condition_labels(subject_id):
    conditions = parse_quest_file(subject_id)
    base = next(item for item in conditions if item['condition'].strip().lower() == 'base')
    if pd.isna(base['stai_mean']):
        raise ValueError(f'S{subject_id}: Base STAI mean cannot be calculated')
    base_mean = base['stai_mean']

    labeled = []
    for item in conditions:
        name = item['condition']
        if name.strip().lower() == 'base':
            continue
        if pd.isna(item['stai_mean']) or pd.isna(item['start_seconds']) or pd.isna(item['end_seconds']):
            continue
        if item['end_seconds'] <= item['start_seconds']:
            raise ValueError(f'S{subject_id}: invalid interval for {name}: {item}')
        item = dict(item)
        item['baseline_stai_mean'] = base_mean
        item['binary_anxiety'] = int(item['stai_mean'] - base_mean >= STAI_INCREASE_THRESHOLD)
        labeled.append(item)

    if not labeled:
        raise ValueError(f'S{subject_id}: no non-Base conditions have both valid STAI scores and timing')
    return labeled


def compute_features_stai(subject_id):
    """Filter and align raw physiological channels to the WESAD 700-Hz timeline."""
    raw_data = load_raw_subject(subject_id)
    wrist = raw_data['signal']['wrist']
    chest = raw_data['signal']['chest']
    labels = np.asarray(raw_data['label']).ravel()

    eda_df = pd.DataFrame(wrist['EDA'], columns=['EDA'])
    bvp_df = pd.DataFrame(wrist['BVP'], columns=['BVP'])
    acc_df = pd.DataFrame(wrist['ACC'], columns=['ACC_x', 'ACC_y', 'ACC_z'])
    temp_df = pd.DataFrame(wrist['TEMP'], columns=['TEMP'])
    acc_c_df = pd.DataFrame(chest['ACC'], columns=['ACC_x_C', 'ACC_y_C', 'ACC_z_C'])
    ecg_df = pd.DataFrame(chest['ECG'], columns=['ECG'])
    eda_c_df = pd.DataFrame(chest['EDA'], columns=['EDA_C'])
    emg_c_df = pd.DataFrame(chest['EMG'], columns=['EMG_C'])
    resp_c_df = pd.DataFrame(chest['Resp'], columns=['Resp_C'])
    temp_c_df = pd.DataFrame(chest['Temp'], columns=['Temp_C'])

    eda_df['EDA'] = butter_lowpass_filter(eda_df['EDA'], 1.0, RAW_FS['EDA'], 6)
    eda_c_df['EDA_C'] = butter_lowpass_filter(eda_c_df['EDA_C'], 1.0, FS['chest'], 6)
    eda_data = nk.eda_phasic(nk.standardize(eda_df['EDA']), sampling_rate=RAW_FS['EDA'])
    eda_df['EDA_SCR'] = eda_data['EDA_Phasic']
    eda_df['EDA_SCL'] = eda_data['EDA_Tonic']
    eda_data_c = nk.eda_phasic(nk.standardize(eda_c_df['EDA_C']), sampling_rate=FS['chest'])
    eda_c_df['EDA_SCR_C'] = eda_data_c['EDA_Phasic']
    eda_c_df['EDA_SCL_C'] = eda_data_c['EDA_Tonic']

    for col in acc_df.columns:
        acc_df[col] = filter_signal_fir(acc_df[col].values)
    for col in acc_c_df.columns:
        acc_c_df[col] = filter_signal_fir(acc_c_df[col].values)

    n_700 = len(labels)
    df = pd.DataFrame(index=np.arange(n_700, dtype=np.int64))
    df['protocol_label'] = labels

    def resample_to_fs(arr, source_fs, target_fs):
        if source_fs == target_fs:
            return np.asarray(arr)
        target_len = int(round(len(arr) * target_fs / source_fs))
        resampled = resample_poly(np.asarray(arr), target_fs, source_fs, axis=0)
        if len(resampled) < target_len:
            pad_width = [(0, target_len - len(resampled))]
            pad_width.extend([(0, 0)] * (resampled.ndim - 1))
            resampled = np.pad(resampled, pad_width, mode='edge')
        return resampled[:target_len]

    def add_signal(name, arr, fs, val_cols=None):
        arr = np.asarray(arr)
        idx = np.round(np.arange(len(arr), dtype=np.int64) * FS['label'] / fs).astype(np.int64)
        idx = np.clip(idx, 0, n_700 - 1)
        if arr.ndim == 1:
            df[name] = pd.Series(arr, index=idx).groupby(level=0).mean()
        else:
            for c_i, col_name in enumerate(val_cols):
                df[col_name] = pd.Series(arr[:, c_i], index=idx).groupby(level=0).mean()

    add_signal('EDA', resample_to_fs(eda_df['EDA'], RAW_FS['EDA'], FS['EDA']), FS['EDA'])
    add_signal('EDA_SCR', resample_to_fs(eda_df['EDA_SCR'], RAW_FS['EDA'], FS['EDA']), FS['EDA'])
    add_signal('EDA_SCL', resample_to_fs(eda_df['EDA_SCL'], RAW_FS['EDA'], FS['EDA']), FS['EDA'])
    add_signal('BVP', resample_to_fs(bvp_df['BVP'], RAW_FS['BVP'], FS['BVP']), FS['BVP'])
    add_signal('TEMP', resample_to_fs(temp_df['TEMP'], RAW_FS['TEMP'], FS['TEMP']), FS['TEMP'])
    add_signal('ACC', acc_df.values, RAW_FS['ACC'], ['ACC_x', 'ACC_y', 'ACC_z'])
    add_signal('ECG', resample_to_fs(ecg_df['ECG'], RAW_FS['ECG'], FS['ECG']), FS['ECG'])
    add_signal('EDA_C', resample_to_fs(eda_c_df['EDA_C'], RAW_FS['chest'], FS['EDA']), FS['EDA'])
    add_signal('EDA_SCR_C', resample_to_fs(eda_c_df['EDA_SCR_C'], RAW_FS['chest'], FS['EDA']), FS['EDA'])
    add_signal('EDA_SCL_C', resample_to_fs(eda_c_df['EDA_SCL_C'], RAW_FS['chest'], FS['EDA']), FS['EDA'])
    add_signal('ACC_C', acc_c_df.values, FS['chest'], ['ACC_x_C', 'ACC_y_C', 'ACC_z_C'])
    add_signal('EMG_C', resample_to_fs(emg_c_df['EMG_C'], RAW_FS['chest'], FS['ECG']), FS['ECG'])
    add_signal('Resp_C', resample_to_fs(resp_c_df['Resp_C'], RAW_FS['chest'], FS['Resp']), FS['Resp'])
    add_signal('Temp_C', resample_to_fs(temp_c_df['Temp_C'], RAW_FS['chest'], FS['TEMP']), FS['TEMP'])

    df.reset_index(names='sample_index', inplace=True)
    return df

print('Signal preprocessing and STAI condition-label functions defined.')

Signal preprocessing and STAI condition-label functions defined.


## 3. Sliding Window Feature Extraction & Saving Pickles

In [43]:
def get_window_stats_ecg(data):
    if 'ECG' not in data or data['ECG'].dropna().empty:
        return {k: None for k in ['bpm', 'ibi', 'sdnn', 'sdsd', 'rmssd', 'pnn20', 'pnn50']}
    try:
        _, measures = hp.process(data['ECG'].dropna().reset_index(drop=True), FS['ECG'])
        return {k: measures[k] for k in ['bpm', 'ibi', 'sdnn', 'sdsd', 'rmssd', 'pnn20', 'pnn50']}
    except Exception:
        return {k: None for k in ['bpm', 'ibi', 'sdnn', 'sdsd', 'rmssd', 'pnn20', 'pnn50']}


ACC_COLUMNS = ['ACC_x', 'ACC_y', 'ACC_z', 'ACC_x_C', 'ACC_y_C', 'ACC_z_C']


def _safe_acc_feature_values(series):
    series = pd.Series(series).dropna()
    if series.empty:
        return np.nan, np.nan, np.nan
    try:
        abs_integral = get_absolute_integral(series)
    except Exception:
        abs_integral = np.nan
    try:
        peak_freq = get_peak_freq(series)
    except Exception:
        peak_freq = np.nan
    return float(series.std()), abs_integral, peak_freq


def aggregate_acc_subwindows(window):
    """Aggregate ACC features from 5-second windows shifted by 0.25 seconds."""
    sub_len = int(round(FS['label'] * ACC_WINDOW_LENGTH_SECONDS))
    sub_step = int(round(FS['label'] * ACC_WINDOW_SHIFT_SECONDS))
    if sub_len <= 0 or sub_step <= 0:
        raise ValueError('ACC sub-window sample length and step must be positive.')

    subwindow_values = {f'{column}_{stat}': [] for column in ACC_COLUMNS for stat in ['std', 'abs_integral', 'peak_freq']}
    subwindow_count = 0
    for start in range(0, len(window) - sub_len + 1, sub_step):
        subwindow = window.iloc[start:start + sub_len]
        subwindow_count += 1
        for column in ACC_COLUMNS:
            std_value, integral_value, peak_value = _safe_acc_feature_values(subwindow[column])
            subwindow_values[f'{column}_std'].append(std_value)
            subwindow_values[f'{column}_abs_integral'].append(integral_value)
            subwindow_values[f'{column}_peak_freq'].append(peak_value)

    if subwindow_count == 0:
        raise ValueError('Main window is shorter than the configured ACC sub-window.')

    aggregated = {}
    for name, values in subwindow_values.items():
        numeric_values = np.asarray(values, dtype=float)
        aggregated[f'ACC_sub_{name}_mean'] = float(np.nanmean(numeric_values))
        aggregated[f'ACC_sub_{name}_std'] = float(np.nanstd(numeric_values))
    aggregated['ACC_subwindow_count'] = subwindow_count
    return aggregated


def extract_stai_features_subject(subject_id):
    df = compute_features_stai(subject_id)
    condition_labels = build_condition_labels(subject_id)
    samples = []
    window_len = int(round(FS['label'] * WINDOW_LENGTH_SECONDS))
    step = int(round(FS['label'] * WINDOW_SHIFT_SECONDS))
    feat_names = None
    candidate_windows = max(0, (len(df) - window_len) // step + 1)
    dropped_unlabeled_windows = 0
    dropped_ambiguous_windows = 0
    acc_subwindow_counts = []

    for start in range(0, len(df) - window_len + 1, step):
        end = start + window_len
        containing = [
            item for item in condition_labels
            if start >= item['start_seconds'] * FS['label']
            and end <= item['end_seconds'] * FS['label']
        ]
        if len(containing) != 1:
            if len(containing) > 1:
                dropped_ambiguous_windows += 1
            else:
                dropped_unlabeled_windows += 1
            continue
        condition = containing[0]
        window = df.iloc[start:end].drop(columns=['protocol_label', 'sample_index']).copy()

        net_acc = get_net_accel(window, 'wrist').rename('net_acc')
        window = pd.concat([net_acc, window], axis=1)
        net_acc_c = get_net_accel(window, 'chest').rename('net_acc_C')
        window = pd.concat([net_acc_c, window], axis=1)
        acc_features = aggregate_acc_subwindows(window)
        acc_subwindow_counts.append(acc_features['ACC_subwindow_count'])

        stats = get_window_stats(window, label=0)
        ecg_stats = get_window_stats_ecg(window)
        if 'label' in stats:
            del stats['label']
        x = pd.DataFrame(stats)
        if feat_names is None:
            feat_names = [f'{row}_{col}' for row in x.index for col in x.columns]
        wdf = pd.DataFrame(x.values.flatten()).T
        wdf.columns = feat_names

        wdf['binary_anxiety'] = condition['binary_anxiety']
        wdf['subject_id'] = subject_id
        wdf['condition'] = condition['condition']
        wdf['window_start_seconds'] = start / FS['label']
        wdf['window_end_seconds'] = end / FS['label']
        wdf['condition_stai_mean'] = condition['stai_mean']
        wdf['baseline_stai_mean'] = condition['baseline_stai_mean']
        for name, value in acc_features.items():
            wdf[name] = value

        wdf['BVP_peak_freq'] = get_peak_freq(window['BVP'].dropna())
        wdf['net_acc_abs_integral'] = get_absolute_integral(window['net_acc'].dropna())
        for col in ACC_COLUMNS:
            wdf[f'{col}_abs_integral'] = get_absolute_integral(window[col].dropna())
            series = window[col].dropna()
            wdf[f'{col}_peak_freq'] = None if series.empty else get_peak_freq(series)
        for key, value in ecg_stats.items():
            wdf[f'ECG_{key}'] = value
        wdf['EDA_slope'] = get_slope(window['EDA'].dropna())
        wdf['EDA_C_slope'] = get_slope(window['EDA_C'].dropna()) if window['EDA_C'].notna().any() else None
        wdf['EDA_drange'] = get_dynamic_range(window['EDA'].dropna())
        wdf['EDA_C_drange'] = get_dynamic_range(window['EDA_C'].dropna()) if window['EDA_C'].notna().any() else None
        wdf['EMG_drange'] = get_dynamic_range(window['EMG_C'].dropna())
        wdf['EMG_abs_integral'] = get_absolute_integral(window['EMG_C'].dropna())
        if window['Resp_C'].notna().any():
            try:
                resp_features = get_resp_features(window['Resp_C'].dropna())
            except (IndexError, ValueError):
                resp_features = (None,) * 6
            for r_name, r_val in zip([
                'Resp_C_rate', 'Resp_C_Inhal_mean', 'Resp_C_Inhal_std',
                'Resp_C_Exhal_mean', 'Resp_C_Exhal_std', 'Resp_C_I/E',
            ], resp_features):
                wdf[r_name] = r_val
        wdf['TEMP_drange'] = get_dynamic_range(window['TEMP'].dropna())
        wdf['TEMP_C_drange'] = get_dynamic_range(window['Temp_C'].dropna()) if window['Temp_C'].notna().any() else None
        wdf['TEMP_slope'] = get_slope(window['TEMP'].dropna())
        wdf['TEMP_C_slope'] = get_slope(window['Temp_C'].dropna()) if window['Temp_C'].notna().any() else None
        samples.append(wdf)

    result = pd.concat(samples, ignore_index=True) if samples else pd.DataFrame()
    result.attrs['candidate_windows'] = candidate_windows
    result.attrs['dropped_unlabeled_windows'] = dropped_unlabeled_windows
    result.attrs['dropped_ambiguous_windows'] = dropped_ambiguous_windows
    result.attrs['acc_subwindow_count'] = int(np.median(acc_subwindow_counts)) if acc_subwindow_counts else 0
    return result


# Process and save subjects to Data/Preprocessed n STAI-labeled/
summary = []
for subject_id in SUBJECTS:
    out_path = OUTPUT_DIR / f'S{subject_id}_stai_60s_shift6s.pkl'
    if out_path.exists() and SKIP_EXTRACTED_SUBJECTS:
        print(f'S{subject_id}: existing file skipped; ensure it was generated with this ACC sub-window version')
        continue
    if out_path.exists() and not OVERWRITE_EXISTING_OUTPUTS:
        print(f'WARNING: {out_path} exists and overwrite is disabled; skipping S{subject_id}.')
        continue

    start_t = time.time()
    features_df = extract_stai_features_subject(subject_id)
    if features_df.empty:
        print(f'S{subject_id}: no fully-contained labeled windows; no file saved')
        continue
    if features_df.filter(regex=r'^ACC_sub_').isna().any().any():
        raise ValueError(f'S{subject_id}: ACC sub-window aggregation produced NaN values')
    features_df.to_pickle(out_path)
    summary.append({
        'subject': subject_id,
        'candidate_windows': features_df.attrs['candidate_windows'],
        'retained_windows': len(features_df),
        'dropped_unlabeled_windows': features_df.attrs['dropped_unlabeled_windows'],
        'dropped_ambiguous_windows': features_df.attrs['dropped_ambiguous_windows'],
        'acc_subwindows_per_main_window': features_df.attrs['acc_subwindow_count'],
        'n_features': len(features_df.columns),
        'no_anxiety_windows': int((features_df['binary_anxiety'] == 0).sum()),
        'anxiety_windows': int((features_df['binary_anxiety'] == 1).sum()),
        'saved_to': out_path.name,
        'seconds': round(time.time() - start_t, 2),
    })
    print(summary[-1])
    del features_df
    gc.collect()

pd.DataFrame(summary)

""


## 4. Preview a Regenerated STAI-Labeled Subject Dataset

Select a subject pickle, inspect its binary label distribution and STAI provenance, and preview rows.

In [44]:
PREVIEW_SUBJECT = 10
preview_path = OUTPUT_DIR / f'S{PREVIEW_SUBJECT}_stai_60s_shift6s.pkl'
if not preview_path.exists():
    raise FileNotFoundError(f'Dataset not found: {preview_path}. Run Section 3 first.')

preview_df = pd.read_pickle(preview_path)
if 'condition_stai_mean' not in preview_df.columns:
    raise ValueError('This pickle lacks STAI provenance and appears to be an old protocol-labeled output. Regenerate it with Section 3.')
print(f'File: {preview_path.name}; shape: {preview_df.shape}')
print('Binary labels (0=No Anxiety, 1=Anxiety):')
display(preview_df['binary_anxiety'].value_counts(dropna=False).sort_index().rename_axis('binary_anxiety').to_frame('windows'))
print('Per-condition STAI means and assigned labels:')
display(preview_df[['condition', 'baseline_stai_mean', 'condition_stai_mean', 'binary_anxiety']].drop_duplicates().sort_values('condition'))
display(preview_df.head())

File: S10_stai_60s_shift6s.pkl; shape: (289, 161)
Binary labels (0=No Anxiety, 1=Anxiety):


,windows
binary_anxiety,
0,289


Per-condition STAI means and assigned labels:


,condition,baseline_stai_mean,condition_stai_mean,binary_anxiety
0,Fun,3.2,1.6,0
56,Medi 1,3.2,2.0,0
229,Medi 2,3.2,2.0,0
110,TSST,3.2,3.0,0


,net_acc_C_mean,net_acc_C_std,net_acc_C_min,net_acc_C_max,net_acc_mean,net_acc_std,net_acc_min,net_acc_max,EDA_mean,EDA_std,...,Resp_C_rate,Resp_C_Inhal_mean,Resp_C_Inhal_std,Resp_C_Exhal_mean,Resp_C_Exhal_std,Resp_C_I/E,TEMP_drange,TEMP_C_drange,TEMP_slope,TEMP_C_slope
0,0.938924,0.002308,0.919657,0.952583,63.006785,0.156195,62.832277,63.417376,0.431408,0.002000,...,NaN,1.775714,0.0,NaN,NaN,inf,1.004163,1.002638,-0.000016,-2.433160e-06
1,0.938985,0.002114,0.924356,0.950952,62.991662,0.126536,62.832277,63.290278,0.431231,0.001891,...,NaN,2.432857,0.0,2.507143,0.0,0.970370,1.003575,1.002506,-0.000014,-1.012605e-06
2,0.939052,0.002100,0.924356,0.950952,63.015146,0.133238,62.832574,63.227306,0.431056,0.002071,...,NaN,NaN,NaN,2.262857,0.0,0.000000,1.003575,1.002506,-0.000012,-8.353151e-07
3,0.939088,0.002095,0.924356,0.950952,63.051354,0.128895,62.832574,63.248793,0.430694,0.002138,...,NaN,2.584286,0.0,2.042857,0.0,1.265035,1.002986,1.002506,-0.000009,-1.254550e-06
4,0.939115,0.001986,0.924356,0.946501,63.078380,0.123141,62.846123,63.248909,0.430180,0.001848,...,NaN,NaN,NaN,3.968571,0.0,0.000000,1.002986,1.002749,-0.000006,-1.953363e-06


## 5. Compare Against the Forked Subject Features

This is a structural comparison only: the forked CSV has WESAD protocol targets, whereas this dataset has STAI-baseline-relative targets. They are different targets and feature-extraction paths, so their target counts or row-wise values are not expected to match.

In [45]:
COMPARISON_SUBJECT = PREVIEW_SUBJECT
stai_path = OUTPUT_DIR / f'S{COMPARISON_SUBJECT}_stai_60s_shift6s.pkl'
forked_path = REPO_ROOT / '_forked' / 'data' / 'subject_feats' / f'S{COMPARISON_SUBJECT}_feats.csv'
if not stai_path.exists() or not forked_path.exists():
    raise FileNotFoundError(f'Missing comparison input: {stai_path} or {forked_path}')

stai_compare_df = pd.read_pickle(stai_path)
forked_compare_df = pd.read_csv(forked_path, index_col=0)
if 'condition_stai_mean' not in stai_compare_df.columns:
    raise ValueError('STAI comparison pickle is stale; regenerate it with Section 3.')

stai_features = set(stai_compare_df.columns) - {
    'binary_anxiety', 'subject_id', 'condition', 'window_start_seconds',
    'window_end_seconds', 'condition_stai_mean', 'baseline_stai_mean',
}
forked_targets = [c for c in forked_compare_df.columns if str(c) in {'0', '1', '2'}]
forked_features = set(forked_compare_df.columns) - set(forked_targets)
common_features = sorted(stai_features & forked_features)
print(f'Subject S{COMPARISON_SUBJECT}: STAI={stai_compare_df.shape}, forked={forked_compare_df.shape}')
print(f'Common feature names: {len(common_features)}')
print(f'STAI label counts:\n{stai_compare_df.binary_anxiety.value_counts().sort_index()}')
print('Shared names do not imply identical numerical features or equivalent labels.')
if common_features:
    display(pd.DataFrame({
        'stai_mean': stai_compare_df[common_features].apply(pd.to_numeric, errors='coerce').mean(),
        'forked_mean': forked_compare_df[common_features].apply(pd.to_numeric, errors='coerce').mean(),
    }))

Subject S10: STAI=(289, 161), forked=(40, 121)
Common feature names: 117
STAI label counts:
0    289
Name: binary_anxiety, dtype: int64
Shared names do not imply identical numerical features or equivalent labels.


,stai_mean,forked_mean
ACC_x_C_abs_integral,29911.085998,21.771808
ACC_x_C_max,0.811264,0.000624
ACC_x_C_mean,0.712169,0.000563
ACC_x_C_min,0.639096,0.000517
ACC_x_C_peak_freq,0.000803,0.001996
...,...,...
net_acc_abs_integral,120771.953230,53.231588
net_acc_max,63.507539,0.054907
net_acc_mean,62.902059,0.030101
net_acc_min,59.665583,0.014139


## 6. Binary Classification Model Training & Evaluation

Train and evaluate models from the generated `S*_stai_60s_shift6s.pkl` feature datasets in `Data/Preprocessed n STAI-labeled 6s/`.

In [46]:
import joblib
from sklearn.ensemble import RandomForestClassifier
from sklearn.metrics import accuracy_score, classification_report, confusion_matrix
from sklearn.model_selection import GroupShuffleSplit
from sklearn.neighbors import KNeighborsClassifier
from sklearn.svm import SVC
from xgboost import XGBClassifier

# Training specifications â€” edit these values before running this cell.
TEST_SIZE = 0.2
RANDOM_STATE = 100
RF_N_ESTIMATORS = 100
SVM_KERNEL = 'rbf'
XGB_N_ESTIMATORS = 100  # boosting rounds, not epochs
KNN_N_NEIGHBORS = 5

pkl_files = sorted(OUTPUT_DIR.glob('S*_stai_60s_shift6s.pkl'))
if not pkl_files:
    raise FileNotFoundError(f'No S*_stai_60s_shift6s.pkl datasets found in {OUTPUT_DIR}. Run Section 3 first.')

all_dfs = []
for path in pkl_files:
    subject_df = pd.read_pickle(path)
    if 'binary_anxiety' not in subject_df.columns or 'subject_id' not in subject_df.columns:
        raise ValueError(f'{path.name} is missing binary_anxiety or subject_id; regenerate it with the current STAI pipeline.')
    all_dfs.append(subject_df)

combined_df = pd.concat(all_dfs, ignore_index=True)
combined_df = combined_df[combined_df['binary_anxiety'].isin([0, 1])].copy()

# Do not feed labels or questionnaire/provenance metadata into the predictors.
metadata_columns = {
    'binary_anxiety', 'subject_id', 'condition', 'window_start_seconds',
    'window_end_seconds', 'condition_stai_mean', 'baseline_stai_mean',
}
feature_columns = [column for column in combined_df.columns if column not in metadata_columns]
X = combined_df[feature_columns].apply(pd.to_numeric, errors='coerce').replace([np.inf, -np.inf], np.nan).fillna(0)
y = combined_df['binary_anxiety'].astype(int)
groups = combined_df['subject_id'].astype(int)

class_counts = y.value_counts().sort_index()
print(f'Loaded {len(pkl_files)} STAI-labeled subjects; combined shape={combined_df.shape}')
print('Class distribution (0=No Anxiety, 1=Anxiety):')
display(class_counts.rename_axis('binary_anxiety').to_frame('windows'))
if y.nunique() != 2:
    raise ValueError(f'Both binary classes are required; got {class_counts.to_dict()}')
if groups.nunique() < 2:
    raise ValueError('At least two subjects are required for subject-grouped evaluation.')

splitter = GroupShuffleSplit(n_splits=1, test_size=TEST_SIZE, random_state=RANDOM_STATE)
train_indices, test_indices = next(splitter.split(X, y, groups=groups))
X_train, X_test = X.iloc[train_indices], X.iloc[test_indices]
y_train, y_test = y.iloc[train_indices], y.iloc[test_indices]
train_subjects = set(groups.iloc[train_indices])
test_subjects = set(groups.iloc[test_indices])
print(f'Train: {X_train.shape}; test: {X_test.shape}')
print(f'Train subjects: {sorted(train_subjects)}; test subjects: {sorted(test_subjects)}')
if train_subjects & test_subjects:
    raise RuntimeError('Subject leakage detected across train/test split.')
if y_train.nunique() != 2 or y_test.nunique() != 2:
    raise ValueError('Grouped split must contain both classes in train and test; adjust TEST_SIZE or the subject grouping.')
if KNN_N_NEIGHBORS > len(X_train):
    raise ValueError('KNN_N_NEIGHBORS exceeds number of training windows.')

models = {
    'Random Forest': RandomForestClassifier(n_estimators=RF_N_ESTIMATORS, random_state=RANDOM_STATE),
    'SVM': SVC(kernel=SVM_KERNEL, random_state=RANDOM_STATE),
    'XGBoost': XGBClassifier(n_estimators=XGB_N_ESTIMATORS, eval_metric='logloss', random_state=RANDOM_STATE),
    'KNN': KNeighborsClassifier(n_neighbors=KNN_N_NEIGHBORS),
}

results = {}
for name, model in models.items():
    print(f'\n--- Training {name} ---')
    model.fit(X_train, y_train)
    y_pred = model.predict(X_test)
    results[name] = {
        'model': model,
        'accuracy': accuracy_score(y_test, y_pred),
        'confusion_matrix': confusion_matrix(y_test, y_pred, labels=[0, 1]),
        'classification_report': classification_report(
            y_test, y_pred, labels=[0, 1], target_names=['No Anxiety', 'Anxiety'], zero_division=0
        ),
    }
    print(f"{name} Accuracy: {results[name]['accuracy']:.4f}")
    print('Confusion Matrix (rows=true, columns=predicted; [0, 1]):\n', results[name]['confusion_matrix'])
    print('Classification Report:\n', results[name]['classification_report'])
    model_filename = MODELS_DIR / f'stai_baseline_relative_{name.lower().replace(" ", "_")}_model.joblib'
    joblib.dump(model, model_filename)
    print(f'Saved model to {model_filename}')

print('\nSTAI-baseline-relative models trained and evaluated.')

Loaded 15 STAI-labeled subjects; combined shape=(4147, 161)
Class distribution (0=No Anxiety, 1=Anxiety):


,windows
binary_anxiety,
0,3404
1,743


Train: (3303, 154); test: (844, 154)
Train subjects: [2, 4, 5, 6, 7, 8, 9, 10, 13, 14, 16, 17]; test subjects: [3, 11, 15]

--- Training Random Forest ---
Random Forest Accuracy: 0.8602
Confusion Matrix (rows=true, columns=predicted; [0, 1]):
 [[581   0]
 [118 145]]
Classification Report:
               precision    recall  f1-score   support

  No Anxiety       0.83      1.00      0.91       581
     Anxiety       1.00      0.55      0.71       263

    accuracy                           0.86       844
   macro avg       0.92      0.78      0.81       844
weighted avg       0.88      0.86      0.85       844

Saved model to e:\Rizqy\Kuliah\Skripsi\repo-skripsi\Models\stai_baseline_relative_random_forest_model.joblib

--- Training SVM ---
SVM Accuracy: 0.6884
Confusion Matrix (rows=true, columns=predicted; [0, 1]):
 [[581   0]
 [263   0]]
Classification Report:
               precision    recall  f1-score   support

  No Anxiety       0.69      1.00      0.82       581
     Anxiety    

## Subject-Independent Evaluation (LOSO)

This cell evaluates every subject as an unseen test subject exactly once. It reports pooled and equal-weight macro-subject metrics.


In [47]:
from sklearn.metrics import accuracy_score, balanced_accuracy_score, confusion_matrix, f1_score, classification_report
from sklearn.model_selection import LeaveOneGroupOut

# Leave-One-Subject-Out: no windows from the held-out subject enter training.
loso_indices = np.arange(len(combined_df))
loso_y = combined_df['binary_anxiety'].astype(int).to_numpy()
loso_groups = combined_df['subject_id'].astype(int).to_numpy()
loso_splitter = LeaveOneGroupOut()
loso_predictions = {
    name: np.full(len(combined_df), -1, dtype=int)
    for name in ['Random Forest', 'SVM', 'XGBoost', 'KNN']
}
loso_subjects = []

for train_idx, test_idx in loso_splitter.split(loso_indices, loso_y, loso_groups):
    held_out_subject = int(combined_df.iloc[test_idx]['subject_id'].iloc[0])
    loso_subjects.append(held_out_subject)
    if np.unique(loso_y[train_idx]).size < 2:
        raise ValueError(f'LOSO training fold for S{held_out_subject} has only one class.')

    loso_models = {
        'Random Forest': RandomForestClassifier(n_estimators=RF_N_ESTIMATORS, random_state=RANDOM_STATE),
        'SVM': SVC(kernel=SVM_KERNEL, random_state=RANDOM_STATE),
        'XGBoost': XGBClassifier(n_estimators=XGB_N_ESTIMATORS, eval_metric='logloss', random_state=RANDOM_STATE),
        'KNN': KNeighborsClassifier(n_neighbors=KNN_N_NEIGHBORS),
    }
    for name, model in loso_models.items():
        model.fit(X.iloc[train_idx], y.iloc[train_idx])
        loso_predictions[name][test_idx] = model.predict(X.iloc[test_idx])

loso_summary = []
for name, predictions in loso_predictions.items():
    if (predictions < 0).any():
        raise RuntimeError(f'LOSO did not produce predictions for {name}.')
    subject_rows = []
    for subject_id in sorted(np.unique(loso_groups)):
        mask = loso_groups == subject_id
        subject_y = loso_y[mask]
        subject_pred = predictions[mask]
        subject_rows.append({
            'subject_id': subject_id,
            'windows': int(mask.sum()),
            'accuracy': accuracy_score(subject_y, subject_pred),
            'anxiety_f1': f1_score(subject_y, subject_pred, labels=[0, 1], average='binary', pos_label=1, zero_division=0),
            'balanced_accuracy': balanced_accuracy_score(subject_y, subject_pred) if np.unique(subject_y).size == 2 else np.nan,
        })
    subject_df = pd.DataFrame(subject_rows)
    loso_summary.append({
        'model': name,
        'pooled_accuracy': accuracy_score(loso_y, predictions),
        'pooled_balanced_accuracy': balanced_accuracy_score(loso_y, predictions),
        'pooled_anxiety_f1': f1_score(loso_y, predictions, zero_division=0),
        'macro_subject_accuracy': subject_df['accuracy'].mean(),
        'macro_subject_balanced_accuracy': subject_df['balanced_accuracy'].mean(),
        'macro_subject_anxiety_f1': subject_df['anxiety_f1'].mean(),
    })
    print(f'\n--- {name} | LOSO ---')
    print('Confusion matrix [rows=true, columns=predicted; 0=No Anxiety, 1=Anxiety]:')
    print(confusion_matrix(loso_y, predictions, labels=[0, 1]))
    print(classification_report(loso_y, predictions, labels=[0, 1], target_names=['No Anxiety', 'Anxiety'], zero_division=0))
    display(subject_df.sort_values('subject_id').round(4))

loso_summary_df = pd.DataFrame(loso_summary).set_index('model')
print('\nLOSO summary:')
display(loso_summary_df.round(4))



--- Random Forest | LOSO ---
Confusion matrix [rows=true, columns=predicted; 0=No Anxiety, 1=Anxiety]:
[[3187  217]
 [ 440  303]]
              precision    recall  f1-score   support

  No Anxiety       0.88      0.94      0.91      3404
     Anxiety       0.58      0.41      0.48       743

    accuracy                           0.84      4147
   macro avg       0.73      0.67      0.69      4147
weighted avg       0.83      0.84      0.83      4147



,subject_id,windows,accuracy,anxiety_f1,balanced_accuracy
0,2,262,1.0000,0.0000,NaN
1,3,266,0.6692,0.6071,0.7179
2,4,274,0.4307,0.0000,0.4958
3,5,275,1.0000,0.0000,NaN
4,6,272,0.9890,0.0000,0.5000
5,7,277,0.6318,0.0000,0.5000
6,8,265,1.0000,0.0000,NaN
7,9,266,1.0000,0.0000,NaN
8,10,289,1.0000,0.0000,NaN
9,11,286,0.6329,0.6708,0.7067



--- SVM | LOSO ---
Confusion matrix [rows=true, columns=predicted; 0=No Anxiety, 1=Anxiety]:
[[3404    0]
 [ 743    0]]
              precision    recall  f1-score   support

  No Anxiety       0.82      1.00      0.90      3404
     Anxiety       0.00      0.00      0.00       743

    accuracy                           0.82      4147
   macro avg       0.41      0.50      0.45      4147
weighted avg       0.67      0.82      0.74      4147



,subject_id,windows,accuracy,anxiety_f1,balanced_accuracy
0,2,262,1.0000,0.0,NaN
1,3,266,0.4135,0.0,0.5
2,4,274,0.4343,0.0,0.5
3,5,275,1.0000,0.0,NaN
4,6,272,0.9890,0.0,0.5
5,7,277,0.6318,0.0,0.5
6,8,265,1.0000,0.0,NaN
7,9,266,1.0000,0.0,NaN
8,10,289,1.0000,0.0,NaN
9,11,286,0.6259,0.0,0.5



--- XGBoost | LOSO ---
Confusion matrix [rows=true, columns=predicted; 0=No Anxiety, 1=Anxiety]:
[[3137  267]
 [ 376  367]]
              precision    recall  f1-score   support

  No Anxiety       0.89      0.92      0.91      3404
     Anxiety       0.58      0.49      0.53       743

    accuracy                           0.84      4147
   macro avg       0.74      0.71      0.72      4147
weighted avg       0.84      0.84      0.84      4147



,subject_id,windows,accuracy,anxiety_f1,balanced_accuracy
0,2,262,1.0000,0.0000,NaN
1,3,266,0.6504,0.5753,0.7019
2,4,274,0.4343,0.0000,0.5000
3,5,275,1.0000,0.0000,NaN
4,6,272,0.9890,0.0000,0.5000
5,7,277,0.6318,0.0000,0.5000
6,8,265,1.0000,0.0000,NaN
7,9,266,1.0000,0.0000,NaN
8,10,289,1.0000,0.0000,NaN
9,11,286,0.5594,0.6294,0.6480



--- KNN | LOSO ---
Confusion matrix [rows=true, columns=predicted; 0=No Anxiety, 1=Anxiety]:
[[2740  664]
 [ 699   44]]
              precision    recall  f1-score   support

  No Anxiety       0.80      0.80      0.80      3404
     Anxiety       0.06      0.06      0.06       743

    accuracy                           0.67      4147
   macro avg       0.43      0.43      0.43      4147
weighted avg       0.67      0.67      0.67      4147



,subject_id,windows,accuracy,anxiety_f1,balanced_accuracy
0,2,262,0.7214,0.0000,NaN
1,3,266,0.1692,0.0000,0.2045
2,4,274,0.5292,0.2873,0.5839
3,5,275,0.8982,0.0000,NaN
4,6,272,0.8419,0.0000,0.4257
5,7,277,0.6282,0.0000,0.4971
6,8,265,0.9019,0.0000,NaN
7,9,266,0.8534,0.0000,NaN
8,10,289,0.9239,0.0000,NaN
9,11,286,0.6224,0.0357,0.5010



LOSO summary:


,pooled_accuracy,pooled_balanced_accuracy,pooled_anxiety_f1,macro_subject_accuracy,macro_subject_balanced_accuracy,macro_subject_anxiety_f1
model,,,,,,
Random Forest,0.8416,0.6720,0.4798,0.8422,0.5974,0.1591
SVM,0.8208,0.5000,0.0000,0.8209,0.5000,0.0000
XGBoost,0.8449,0.7078,0.5330,0.8459,0.6216,0.1775
KNN,0.6713,0.4321,0.0606,0.6713,0.4126,0.0364


## Interpretation of Evaluation Metrics

The evaluation output above contains the following metrics. In this notebook, class `0` means **No Anxiety** and class `1` means **Anxiety**.

- **Accuracy**: the proportion of all evaluated windows that were classified correctly. It is calculated as `(true negatives + true positives) / all windows`. Accuracy can look high when the classes are imbalanced, so it should not be interpreted alone.
- **Precision**: among windows predicted as Anxiety, the proportion that truly are Anxiety. High precision means few false positives.
- **Recall** (also called sensitivity or true-positive rate): among windows that truly are Anxiety, the proportion detected by the model. High recall means few false negatives.
- **F1-score**: the harmonic mean of precision and recall: `2 * precision * recall / (precision + recall)`. It is useful when both false positives and false negatives matter, especially for the Anxiety class.
- **Support**: the number of ground-truth windows belonging to each class in the evaluated test set.
- **Macro average**: the unweighted average of a metric across the two classes. Each class receives equal weight, regardless of how many windows it contains.
- **Weighted average**: the average of a metric weighted by each class's support. The majority class therefore contributes more to this value.
- **Confusion matrix**: a table whose rows are the true classes and whose columns are the predicted classes, in the order `[0, 1]`: `[[true No Anxiety, predicted No Anxiety], [true Anxiety, predicted Anxiety]]`. The diagonal contains correct predictions; the off-diagonal cells are errors.

### Metrics for subject-independent evaluation

- **Pooled accuracy / pooled balanced accuracy**: calculated after combining predictions from all held-out subjects. For pooled accuracy, `correct windows across all subjects / all windows across all subjects`; subjects with more windows contribute more to the result.
- **Macro subject accuracy / macro subject balanced accuracy**: first calculates the metric separately for each held-out subject, then averages those subject scores: `mean(subject 1 score, subject 2 score, ..., subject N score)`. Every subject contributes equally, even when subjects have different numbers of windows. This is the preferred measure when the goal is performance on a new person.

### Pooled accuracy versus macro subject accuracy

These metrics answer different questions. **Pooled accuracy** asks: *“What proportion of all windows, considered together, were classified correctly?”* **Macro subject accuracy** asks: *“What is the average accuracy for a typical subject, giving every subject equal weight?”* They differ whenever subjects contribute different numbers of windows or have different error rates. For example, if Subject A has 900 windows with 90% accuracy and Subject B has 100 windows with 50% accuracy, pooled accuracy is `(900 × 0.90 + 100 × 0.50) / 1000 = 86%`, whereas macro subject accuracy is `(90% + 50%) / 2 = 70%`. The pooled result is dominated by Subject A; the macro subject result exposes that performance is poorer for Subject B.
- **Anxiety F1**: the F1-score for class `1` only. It summarizes how well the model detects the STAI-defined Anxiety windows.
- **Balanced accuracy**: the average recall of the two classes: `(recall No Anxiety + recall Anxiety) / 2`. It prevents the majority class from dominating the score. For a subject with only one ground-truth class, subject-level balanced accuracy is not defined and is excluded from the macro average.

The random window split is retained only for comparison with the reference pipeline. The Leave-One-Subject-Out (LOSO) results are the more reliable estimate of performance on previously unseen subjects because overlapping windows from one subject never appear in both training and test data.
